# EDA — Expedientes INDECOPI (2017–2021)
Dataset generado del merge entre expedientes presentados y resueltos ante la Sala Especializada en Protección al Consumidor (SPC).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv('expedientes_merged.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 1. Estructura general

In [ ]:
df.dtypes

In [ ]:
# Nulos por columna
nulls = df.isnull().sum().sort_values(ascending=False)
null_pct = (nulls / len(df) * 100).round(1)
pd.DataFrame({'nulos': nulls, '%': null_pct})

## 2. Distribución temporal — expedientes presentados por año

In [ ]:
por_año = df['año_pres'].value_counts().sort_index()

fig, ax = plt.subplots()
bars = ax.bar(por_año.index.astype(str), por_año.values, color='steelblue', width=0.6)
ax.bar_label(bars, padding=3)
ax.set_title('Expedientes presentados por año')
ax.set_xlabel('Año')
ax.set_ylabel('Cantidad')
plt.tight_layout()
plt.show()

por_año

## 3. Tipo de expediente

In [ ]:
tipo = df['TIPO_EXPEDIENTE_pres'].value_counts()

fig, ax = plt.subplots()
bars = ax.barh(tipo.index[::-1], tipo.values[::-1], color='steelblue')
ax.bar_label(bars, padding=3)
ax.set_title('Tipo de expediente')
ax.set_xlabel('Cantidad')
plt.tight_layout()
plt.show()

print(f'APELACION representa el {tipo["APELACION"]/len(df)*100:.1f}% del total')

## 4. Top 10 materias más frecuentes

In [ ]:
materia = df['MATERIA_pres'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(materia.index[::-1], materia.values[::-1], color='coral')
ax.bar_label(bars, padding=3)
ax.set_title('Top 10 materias (expedientes presentados)')
ax.set_xlabel('Cantidad')
plt.tight_layout()
plt.show()

## 5. Match con resoluciones

In [ ]:
match = df['RES_MATCH_SOURCE'].value_counts()
labels = {'nro': 'Match por NRO_EXPEDIENTE', 'origen': 'Match por EXPEDIENTE_ORIGEN', 'none': 'Sin match'}
match.index = [labels.get(i, i) for i in match.index]

fig, ax = plt.subplots(figsize=(6, 4))
match.plot.pie(ax=ax, autopct='%1.1f%%', colors=['steelblue','orange','lightgray'], startangle=90)
ax.set_ylabel('')
ax.set_title('Fuente del match con resoluciones')
plt.tight_layout()
plt.show()

match

## 6. Forma de conclusión (top 10)

In [ ]:
# Limpiamos el prefijo '>>' para visualizar mejor
df['FORMA_CONCLUSION_clean'] = df['FORMA_CONCLUSION'].str.replace('>>', '', regex=False).str.split('\n').str[0].str.strip()
forma = df['FORMA_CONCLUSION_clean'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(forma.index[::-1], forma.values[::-1], color='mediumseagreen')
ax.bar_label(bars, padding=3)
ax.set_title('Top 10 formas de conclusión')
ax.set_xlabel('Cantidad')
plt.tight_layout()
plt.show()

## 7. Tiempo de resolución (días)

In [ ]:
df['FECHA_PRESENTACION_pres'] = pd.to_datetime(df['FECHA_PRESENTACION_pres'], errors='coerce')
df['FECHA_RESOLUCION'] = pd.to_datetime(df['FECHA_RESOLUCION'], errors='coerce')
df['DIAS_RESOLUCION'] = (df['FECHA_RESOLUCION'] - df['FECHA_PRESENTACION_pres']).dt.days

dias = df['DIAS_RESOLUCION'].dropna()
dias_valid = dias[(dias >= 0) & (dias < 1500)]  # descartamos outliers absurdos

print(dias_valid.describe().round(1))

fig, ax = plt.subplots()
ax.hist(dias_valid, bins=40, color='steelblue', edgecolor='white')
ax.axvline(dias_valid.median(), color='red', linestyle='--', label=f'Mediana: {dias_valid.median():.0f} días')
ax.set_title('Distribución de días hasta resolución')
ax.set_xlabel('Días')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Resumen final

In [ ]:
resueltos = df['RES_MATCH_SOURCE'].ne('none').sum()
print(f'Total expedientes:          {len(df):,}')
print(f'Con resolución encontrada:  {resueltos:,} ({resueltos/len(df)*100:.1f}%)')
print(f'Sin resolución (none):      {(df["RES_MATCH_SOURCE"]=="none").sum():,}')
print(f'Rango temporal:             {df["año_pres"].min()}–{df["año_pres"].max()}')
print(f'Tipos de expediente únicos: {df["TIPO_EXPEDIENTE_pres"].nunique()}')
print(f'Materias únicas:            {df["MATERIA_pres"].nunique()}')
print(f'Mediana días resolución:    {dias_valid.median():.0f} días')